In [1]:
!pip install fastapi uvicorn pydantic

In [1]:
from fastapi import FastAPI
import uvicorn
import threading
from pydantic import BaseModel
from typing import List,Optional
from uuid import UUID,uuid4
app = FastAPI()


tasks=[] #inreality connect this to database

class Task(BaseModel): #reprsent object that will passed over api
  id:Optional[UUID]=None
  title:str
  description:Optional[str]=None
  priority:int=1
  complete:bool=False


@app.post("/tasks/",response_model=Task)
def create_task(task:Task):
  task.id=uuid4()
  tasks.append(task)
  return task
@app.get("/tasks/",response_model=List[Task])
def read_tasks():
    return tasks

@app.get("/tasks/{task_id}",response_model=Task)#when this end point is called
#this function will run
def read_task(task_id:UUID):
    for task in tasks:
        if task.id==task_id:
            return task
    raise HTTPException(status_code=404,detail="Task not found ")

@app.delete("/tasks/{task_id}",response_model=Task)
def delete_task(task_id:UUID):
 for idx,task in enumerate(tasks):
  if task.id==task_id:
    return tasks.pop(idx)
  raise HTTPException(status_code=404,detail="Task Not Fond" )
from fastapi import HTTPException

@app.put("/tasks/{task_id}", response_model=Task)
def update_task(task_id: UUID, updated_task: Task):

    for idx, task in enumerate(tasks):
        if task.id == task_id:

            # keep old task, overwrite with new values
            updated_task.id = task_id
            tasks[idx] = updated_task

            return updated_task

    raise HTTPException(status_code=404, detail="Task not found")

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run)
thread.start()

In [2]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

In [3]:
!./cloudflared-linux-amd64 tunnel --url http://localhost:8000 --protocol http2

2026-04-27T11:23:21Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-27T11:23:21Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-27T11:23:23Z INF +--------------------------------------------------------------------------------------------+
2026-04-27T11:23:23Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-27T11:23:23Z INF |  https://dash-ladder-geographic-arbitrary.trycloudflar

NameError: name '__name' is not defined